# RunnableSequence runnable

In [ ]:
# -----------------------------
# Imports
# -----------------------------
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough
from dotenv import load_dotenv
import os

# -----------------------------
# Load environment variables
# -----------------------------
load_dotenv()

classification_schema = {
  "title": "ClassificationResult",
  "type": "object",
  "properties": {
    "category": {
      "title": "Category",
      "description": "The category of the support ticket.",
      "type": "string",
      "enum": ["billing", "technical", "account"]
    }
  },
  "required": ["category"]
}

# -----------------------------
# Initialize Ollama model
# -----------------------------
model = ChatOllama(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER"),
    base_url=os.getenv("REVERSE_PROXY_HOST"),
    temperature=0
)

structured_model = model.with_structured_output(classification_schema)

parser = StrOutputParser()


# -----------------------------
# Step 1: Classify support ticket
# -----------------------------
classification_prompt = PromptTemplate(
    template="""
Classify the following customer support message into one category:
billing, technical, or account.

Message:
{message}
""",
    input_variables=["message"]
)


# -----------------------------
# RunnableSequence pipeline
# -----------------------------
chain = RunnableSequence(
    classification_prompt,
    structured_model
)
# chain = classification_prompt | model # --- IGNORE ---

# -----------------------------
# Run example
# -----------------------------
result = chain.invoke({
    "message": "I was charged twice for my subscription this month."
})

print(result['category'])

# RunnableBranch runnable

In [ ]:
# -----------------------------
# Imports
# -----------------------------
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

# -----------------------------
# Load environment variables
# -----------------------------
load_dotenv()

# -----------------------------
# Initialize model
# -----------------------------
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)

parser = StrOutputParser()

# -----------------------------
# Expert prompts
# -----------------------------
math_prompt = PromptTemplate.from_template(
"""
You are a math expert.

Solve this problem:

{question}
"""
)

code_prompt = PromptTemplate.from_template(
"""
You are a programming expert.

Answer this coding question:

{question}
"""
)

general_prompt = PromptTemplate.from_template(
"""
Answer this question clearly:

{question}
"""
)

# -----------------------------
# Expert chains
# -----------------------------
math_chain = math_prompt | model | parser
code_chain = code_prompt | model | parser
general_chain = general_prompt | model | parser

# -----------------------------
# RunnableBranch router
# -----------------------------
chain = RunnableBranch(

    # Condition → runnable
    (lambda x: "calculate" in x["question"].lower(), math_chain),

    (lambda x: "python" in x["question"].lower(), code_chain),

    # Default fallback
    general_chain
)

# -----------------------------
# Run example
# -----------------------------
result = chain.invoke({
    "question": "How do I create a loop in Python?"
})

print(result)

# RunnableLambda runnable

In [ ]:
# -----------------------------
# Imports
# -----------------------------
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

load_dotenv()

# Initialize model
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)

parser = StrOutputParser()

# Function to preprocess input
def clean_text(x):
    return {"text": x["text"].strip().lower()}

cleaner = RunnableLambda(clean_text)
# cleaner = RunnableLambda(lambda x: {"text": x["text"].strip().lower()}) --- IGNORE ---

prompt = PromptTemplate.from_template(
"""
Explain the following topic:

{text}
"""
)

chain = cleaner | prompt | model | parser

result = chain.invoke({
    "text": "   What is a Hash Table?   "
})

print(result)

# RunnableParallel runnable

In [ ]:
# -----------------------------
# Imports
# -----------------------------
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)

sentiment_prompt = PromptTemplate.from_template(
"Determine sentiment of this review: {review}"
)

topic_prompt = PromptTemplate.from_template(
"Identify the topic of this review: {review}"
)

chain = RunnableParallel(
    sentiment=sentiment_prompt | model | parser,
    topic=topic_prompt | model | parser
)

result = chain.invoke({
    "review": "The camera is great but battery life is poor."
})

print(result)

# RunnablePassthrough runnable

In [ ]:
# -----------------------------
# Imports
# -----------------------------
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# -----------------------------
# Load environment variables
# -----------------------------
load_dotenv()

# -----------------------------
# Initialize model
# -----------------------------
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)

parser = StrOutputParser()


# -----------------------------
# Prompts
# -----------------------------
sentiment_prompt = PromptTemplate.from_template(
"""
Determine the sentiment of this review.

Review: {review}
"""
)

issue_prompt = PromptTemplate.from_template(
"""
Identify the main issue mentioned in this review.

Review: {review}
"""
)

summary_prompt = PromptTemplate.from_template(
"""
Original Review:
{review}

Sentiment:
{sentiment}

Main Issue:
{issue}

Write a final one-sentence summary.
"""
)


# -----------------------------
# Step 1: Parallel analysis
# -----------------------------
parallel_analysis = RunnableParallel(
    sentiment=sentiment_prompt | model | parser,
    issue=issue_prompt | model | parser,
    review=RunnablePassthrough()  # keep original review
)


# -----------------------------
# Step 2: Sequential step
# -----------------------------
chain = parallel_analysis | summary_prompt | model | parser


# -----------------------------
# Run example
# -----------------------------
result = chain.invoke({
    "review": "The phone camera is amazing but the battery drains very quickly."
})

print(result)

# Runnable DEMO

In [ ]:
# -----------------------------
# Imports
# -----------------------------
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableAssign
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from typing import Literal

# -----------------------------
# Load env
# -----------------------------
load_dotenv()

# -----------------------------
# Model
# -----------------------------
model = ChatOllama(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("REVERSE_PROXY_HOST"),
    temperature=0
)

parser = StrOutputParser()

# -----------------------------
# Pydantic Models
# -----------------------------
class ClassificationResult(BaseModel):
    category: Literal["billing", "technical", "account"] = Field(
        description="Support ticket category"
    )


class UserInfo(BaseModel):
    name: str
    email: str
    subscription_level: Literal["free", "premium", "enterprise"]
    account_number: str


# -----------------------------
# Step 1: Classification Prompt
# -----------------------------
classification_prompt = PromptTemplate(
    template="""
Classify the following support message into one category:
billing, technical, or account.

Message:
{message}
""",
    input_variables=["message"]
)

classification_chain = (
    classification_prompt
    | model.with_structured_output(ClassificationResult)
)

# -----------------------------
# Step 2: Extract User Info
# -----------------------------
user_info_prompt = PromptTemplate(
    template="""
Extract the following user information from this support ticket.

Message:
{message}

Return:
- name
- email
- subscription_level (free, premium, enterprise)
- account_number
""",
    input_variables=["message"]
)

user_info_chain = (
    user_info_prompt
    | model.with_structured_output(UserInfo)
)

# -----------------------------
# Step 3: Generate Response
# -----------------------------
response_prompt = PromptTemplate(
    template="""
Customer support category: {category}

Customer information:
Name: {name}
Email: {email}
Subscription: {subscription_level}
Account Number: {account_number}

Customer query:
{query}

Write a short professional support response helping the customer.

Rules:
- Do NOT include an email subject.
- Do NOT include placeholders like [Company Name] or [Your Name].
- Address the customer by name.

""",
    input_variables=[
        "category",
        "name",
        "email",
        "subscription_level",
        "account_number",
        "query"
    ]
)

response_chain = response_prompt | model | parser

# -----------------------------
# Combine Pipelines
# -----------------------------
pipeline = RunnableParallel(
    classification=classification_chain,
    user=user_info_chain,
    query=RunnablePassthrough()
)

# -----------------------------
# Merge outputs before response
# -----------------------------
# RunnableAssign: adds or transforms fields in the flowing dictionary
# without removing existing ones. Useful for reshaping data between steps.
prepare_response_inputs = RunnableAssign({
    "category": lambda x: x["classification"].category,
    "name": lambda x: x["user"].name,
    "email": lambda x: x["user"].email,
    "subscription_level": lambda x: x["user"].subscription_level,
    "account_number": lambda x: x["user"].account_number,
    "query": lambda x: x["query"]["message"]
})
chain = pipeline | prepare_response_inputs | response_chain

# -----------------------------
# Example
# -----------------------------
result = chain.invoke({
    "message": """
Hello, my name is John Doe.
My email is john@example.com.
Account number: ACC123.
I was charged twice for my premium subscription.
"""
})

print(result)